# 11. Multi-World Systems

A `System` gathers several worlds and the orbits that connect them. Each orbiting world names its tidal host, the body that raises its tides, and carries its semi-major axis and eccentricity about that host. One world is the star that lights them, often the same body as the tidal host. From that, the system computes each world's instantaneous orbital and spin evolution in one call.

This notebook assembles a small planetary system, reads the whole system's evolution, changes a world's tidal host, adds a world, and saves the result.


In [1]:
%matplotlib inline
import numpy as np

from TidalPy.constants import G
from TidalPy.structures_x.system import System
from TidalPy.structures_x.worlds.stellar import StarWorld
from TidalPy.structures_x.configs import build_world
from TidalPy.Tides_x.classes import make_tide

AU = 1.495978707e11

def tidal_planet(config):
    "Build a world and give it a fixed-Q tide so it can dissipate."
    world = build_world(config)
    world.set_tide_model(make_tide("fixed_q", {"fixed_k": [0.3], "fixed_q": [100.0]}))
    world.set_tide_config(min_degree_l=2, max_degree_l=2, eccentricity_truncation=2, obliquity_truncation=0)
    return world

star = StarWorld("Sun", 6.957e8, 1.988e30)
star.set_effective_temperature(5772.0)

inner = tidal_planet({"schema_version": "0.2.0", "name": "Inner", "type": "terrestrial",
                      "radius_m": 6.0e6, "mass_kg": 5.0e24, "spin_frequency_rad_s": 2.0e-5,
                      "layers": {"core": {"class": "physics", "type": "iron", "layer_index": 0,
                                          "radius_outer_m": 3.0e6, "is_tidal": False},
                                 "mantle": {"class": "solidliquid", "type": "mantle_rock", "layer_index": 1,
                                            "radius_fraction": 1.0, "is_tidal": True}}})
outer = tidal_planet({"schema_version": "0.2.0", "name": "Outer", "type": "gasgiant",
                      "radius_m": 6.0e7, "mass_kg": 6.0e26, "spin_frequency_rad_s": 1.0e-4,
                      "layers": {"envelope": {"class": "gas", "type": "gas", "layer_index": 0,
                                              "radius_fraction": 1.0, "is_tidal": True}}})

system = System("Demo-System")
system.add_world(star, is_star=True)
system.add_world(inner, tidal_host=star, semi_major_axis=0.20 * AU, eccentricity=0.05)
system.add_world(outer, tidal_host=star, semi_major_axis=0.80 * AU, eccentricity=0.10)
for w in (inner, outer):
    system.set_stellar_semi_major_axis(w, system.get_semi_major_axis(w))
    w.set_spin_frequency(system.calc_orbital_frequency(w))   # start synchronous: spin = mean motion

print(f"{system.num_worlds} worlds: {[w.name for w in system]}")
print(f"star = {system.star.name}, tidal hosts: " +
      ", ".join(f"{w.name} -> {system.get_tidal_host(w).name}" for w in (inner, outer)))


3 worlds: ['Sun', 'Inner', 'Outer']
star = Sun, tidal hosts: Inner -> Sun, Outer -> Sun


## System Evolution

`calc_system_evolution` returns one record per world with its instantaneous tidal heating and orbital and spin rates. The Sun has no tidal host, so nothing forces it: its record has `evolved` set to `False` and zero rates. Both planets start spinning synchronously on an eccentric orbit, so `dspin/dt` comes out positive: the tide spins them up toward the faster pseudo-synchronous rate that an eccentric orbit settles at.

In [2]:
rows = system.calc_system_evolution()
print(f"{'world':8} {'heating (W)':>12} {'da/dt (m/s)':>13} {'de/dt (1/s)':>13} {'dspin/dt (1/s^2)':>17}")
for r in rows:
    name = system.worlds[r["world_index"]].name
    print(f"{name:8} {r['tidal_heating']:12.3e} {r['da_dt']:13.3e} {r['de_dt']:13.3e} {r['dspin_dt']:17.3e}")


world     heating (W)   da/dt (m/s)   de/dt (1/s)  dspin/dt (1/s^2)
Sun         0.000e+00     0.000e+00     0.000e+00         0.000e+00
Inner       5.147e+11    -3.743e-12    -4.621e-22         5.442e-21
Outer       6.780e+12    -6.441e-12    -1.001e-22         4.626e-23


## Reading and Setting Orbital Elements

Orbital elements can be read and changed at any time. Changing an eccentricity immediately changes that world's dissipation, since tidal heating grows with eccentricity.

In [3]:
print("before:", system.get_eccentricity(inner), "->", end=" ")
system.set_eccentricity(inner, 0.10)
print(system.get_eccentricity(inner))

heating_before = [r for r in system.calc_system_evolution() if system.worlds[r["world_index"]].name == "Inner"][0]
print(f"Inner heating at e=0.10: {heating_before['tidal_heating']:.3e} W")


before: 0.05 -> 0.1
Inner heating at e=0.10: 2.221e+12 W


## Changing Roles

A world's tidal host is changed with `set_tidal_host`, and `set_star` moves the star role. Changing a world's tidal host changes what that world is considered to orbit: its stored orbital elements are then interpreted about the new host. Only reassign when the new arrangement is physically sensible. The cell below makes the outer planet the inner planet's tidal host and then restores the Sun.

In [4]:
print("Inner's tidal host:", system.get_tidal_host(inner).name, "| star:", system.star.name)
system.set_tidal_host(inner, outer)
print("after set_tidal_host(inner, outer):", system.get_tidal_host(inner).name, "| star:", system.star.name)
system.set_tidal_host(inner, star)
print("restored:", system.get_tidal_host(inner).name, "| star:", system.star.name)

Inner's tidal host: Sun | star: Sun
after set_tidal_host(inner, outer): Outer | star: Sun
restored: Sun | star: Sun


## Adding and Removing Worlds

More worlds can be added at any time with `add_world`. There is no in-place remove. To drop a world, rebuild the system from the ones you want to keep.

In [5]:
third = tidal_planet({"schema_version": "0.2.0", "name": "Third", "type": "terrestrial",
                      "radius_m": 4.0e6, "mass_kg": 2.0e24, "spin_frequency_rad_s": 3.0e-5,
                      "layers": {"mantle": {"class": "solidliquid", "type": "mantle_rock", "layer_index": 0,
                                            "radius_fraction": 1.0, "is_tidal": True}}})
system.add_world(third, tidal_host=star, semi_major_axis=0.50 * AU, eccentricity=0.02)
system.set_stellar_semi_major_axis(third, system.get_semi_major_axis(third))
print(f"now {system.num_worlds} worlds: {[w.name for w in system]}")


now 4 worlds: ['Sun', 'Inner', 'Outer', 'Third']


## Saving the System

A complete system saves to TOML (a readable recipe) or to a binary snapshot, as in the Basics notebook. The binary form restores each world as its concrete type.

In [6]:
import tempfile
from pathlib import Path
from TidalPy.structures_x.system import System

work = Path(tempfile.mkdtemp(prefix="tidalpy_sys_"))
system.save_binary(str(work / "system.tpyb"))
reloaded = System()
reloaded.load_binary(str(work / "system.tpyb"))
print("reloaded:", [(w.name, type(w).__name__) for w in reloaded])

import shutil
shutil.rmtree(work, ignore_errors=True)


reloaded: [('Sun', 'StarWorld'), ('Inner', 'LayeredWorld'), ('Outer', 'GasGiantWorld'), ('Third', 'LayeredWorld')]
